# Assignment 03 — Bài toán 3: Khai phá Sở thích Khách hàng TMĐT (Tiki) — Phân loại đa lớp

**Môn học:** Intelligent System Development — TS. Trần Đình Quế
**Sinh viên:** Đinh Hải Triều — B23DCCN843 — Lớp 06

---

## Mục tiêu

1. Cài đặt **mạng nơ-ron sâu 5 tầng** thuần **NumPy** cho **phân loại đa lớp**
   (tầng 5 dùng Softmax + Categorical Cross-Entropy) và bản **PyTorch** đối chiếu.
2. Kết hợp **đặc trưng bảng** (tuổi, điểm đánh giá, lượt hữu ích) với **đặc trưng
   văn bản TF-IDF** từ nội dung đánh giá của khách hàng.
3. Đối sánh với Logistic Regression, Naive Bayes, Decision Tree, Random Forest.
4. Xuất trọng số ra `public/model_deep.json` để web app React chạy suy luận
   100% phía trình duyệt trên Vercel.

**Bài toán:** Multiclass Classification — từ hành vi + bình luận của khách hàng,
dự đoán **nhóm ngành hàng họ đang quan tâm** (`Department Name`, 6 lớp).

> *Ghi chú về dữ liệu:* bộ dữ liệu gốc là **Women's Clothing E-Commerce Reviews**
> (23.486 đánh giá thật). Bài toán được đặt theo đúng ngữ cảnh sàn TMĐT Việt Nam
> (Tiki): mỗi bản ghi là một khách hàng để lại đánh giá, và mục tiêu là suy ra
> **sở thích ngành hàng** của họ để phục vụ gợi ý sản phẩm.

In [1]:
import json
import time
import pathlib

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report,
    balanced_accuracy_score, top_k_accuracy_score,
)

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 150, "font.family": "DejaVu Sans",
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
})

ROOT = pathlib.Path.cwd()
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
print("Thư mục làm việc:", ROOT)

Thư mục làm việc: C:\Users\admin\Downloads\bt-thay quế\tuan 2\customer-behavior-predict


## 1. Nạp dữ liệu và khảo sát

In [2]:
df = pd.read_csv(ROOT / "ml" / "data" / "ecommerce_raw.csv", index_col=0)
print("Kích thước gốc:", df.shape)
print("\nSố giá trị thiếu theo cột:")
print(df.isna().sum().to_string())

TARGET = "Department Name"
df = df.dropna(subset=[TARGET]).reset_index(drop=True)
df["Review Text"] = df["Review Text"].fillna("")
print(f"\nSau khi loại bản ghi thiếu nhãn: {df.shape}")
df.head(3)

Kích thước gốc: (23486, 10)

Số giá trị thiếu theo cột:
Clothing ID                   0
Age                           0
Title                      3810
Review Text                 845
Rating                        0
Recommended IND               0
Positive Feedback Count       0
Division Name                14
Department Name              14
Class Name                   14

Sau khi loại bản ghi thiếu nhãn: (23472, 10)


,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses


In [3]:
VI_NAME = {
    "Tops": "Áo (Tops)",
    "Dresses": "Váy đầm (Dresses)",
    "Bottoms": "Quần (Bottoms)",
    "Intimate": "Đồ mặc trong (Intimate)",
    "Jackets": "Áo khoác (Jackets)",
    "Trend": "Hàng xu hướng (Trend)",
}
dist = df[TARGET].value_counts()
dist_df = pd.DataFrame({
    "Số mẫu": dist,
    "Tỉ lệ (%)": (100 * dist / len(df)).round(2),
    "Tên tiếng Việt": [VI_NAME[k] for k in dist.index],
})
print(f"Tỉ lệ mất cân bằng lớp lớn nhất / nhỏ nhất: {dist.max()/dist.min():.1f} lần")
dist_df

Tỉ lệ mất cân bằng lớp lớn nhất / nhỏ nhất: 88.0 lần


,Số mẫu,Tỉ lệ (%),Tên tiếng Việt
Department Name,,,
Tops,10468,44.60,Áo (Tops)
Dresses,6319,26.92,Váy đầm (Dresses)
Bottoms,3799,16.19,Quần (Bottoms)
Intimate,1735,7.39,Đồ mặc trong (Intimate)
Jackets,1032,4.40,Áo khoác (Jackets)
Trend,119,0.51,Hàng xu hướng (Trend)


### 1.1. Cảnh báo rò rỉ nhãn: cột `Class Name`

`Class Name` (20 giá trị: Blouses, Jeans, Dresses, ...) là **hạng mục con** của
`Department Name`. Ánh xạ Class → Department là **một–một xác định**: biết
"Jeans" thì chắc chắn Department = "Bottoms". Nếu đưa cột này vào đầu vào, mô
hình đạt ~100% accuracy nhưng **không học được gì** — đó là rò rỉ nhãn kinh điển.
Ta **loại bỏ hoàn toàn** `Class Name`.

In [4]:
leak = df.groupby("Class Name", observed=True)[TARGET].nunique()
print(f"Số Class Name ánh xạ tới đúng 1 Department: {(leak == 1).sum()}/{len(leak)}")
print("=> Class Name xác định hoàn toàn nhãn -> BẮT BUỘC loại bỏ.\n")
print(df.groupby("Class Name", observed=True)[TARGET].agg(lambda s: s.iloc[0]).head(10).to_string())

Số Class Name ánh xạ tới đúng 1 Department: 20/20
=> Class Name xác định hoàn toàn nhãn -> BẮT BUỘC loại bỏ.

Class Name
Blouses               Tops
Casual bottoms     Bottoms
Chemises          Intimate
Dresses            Dresses
Fine gauge            Tops
Intimates         Intimate
Jackets            Jackets
Jeans              Bottoms
Knits                 Tops
Layering          Intimate


In [5]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# (a) phân bố lớp
cols = plt.cm.viridis(np.linspace(0.15, 0.9, len(dist)))
bars = axes[0, 0].barh([VI_NAME[k] for k in dist.index][::-1], dist.values[::-1], color=cols[::-1])
for b, v in zip(bars, dist.values[::-1]):
    axes[0, 0].text(v + 120, b.get_y() + b.get_height() / 2, f"{v:,}", va="center", fontsize=8.5)
axes[0, 0].set_title("(a) Phân bố 6 lớp — mất cân bằng nghiêm trọng (88:1)", fontweight="bold")
axes[0, 0].set_xlabel("Số đánh giá"); axes[0, 0].set_xlim(0, dist.max() * 1.18)

# (b) độ dài bình luận
lens = df["Review Text"].str.split().str.len()
axes[0, 1].hist(lens, bins=60, color="#8b5cf6", edgecolor="white")
axes[0, 1].axvline(lens.median(), color="#111827", ls="--", label=f"Trung vị = {lens.median():.0f} từ")
axes[0, 1].set_title("(b) Độ dài bình luận (số từ)", fontweight="bold")
axes[0, 1].set_xlabel("Số từ"); axes[0, 1].set_ylabel("Số đánh giá"); axes[0, 1].legend()

# (c) rating theo lớp
sns.boxplot(data=df, y=TARGET, x="Rating", order=dist.index, ax=axes[1, 0],
            palette="viridis", hue=TARGET, legend=False)
axes[1, 0].set_title("(c) Rating gần như giống nhau ở mọi lớp\n=> đặc trưng bảng đơn lẻ không phân biệt được",
                     fontweight="bold", fontsize=10)
axes[1, 0].set_ylabel(""); axes[1, 0].set_xlabel("Rating (1–5)")

# (d) tuổi theo lớp
sns.violinplot(data=df, y=TARGET, x="Age", order=dist.index, ax=axes[1, 1],
               palette="viridis", hue=TARGET, legend=False, cut=0)
axes[1, 1].set_title("(d) Phân bố tuổi theo lớp — cũng chồng lấn gần hoàn toàn",
                     fontweight="bold", fontsize=10)
axes[1, 1].set_ylabel(""); axes[1, 1].set_xlabel("Tuổi")

plt.tight_layout()
plt.savefig(FIG / "p3_fig1_eda.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_20440\2126034408.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Xây dựng không gian đặc trưng

| Khối | Nguồn | Số chiều | Xử lý |
|---|---|---|---|
| Số | Age, Rating, Positive Feedback Count | 3 | median-impute → StandardScaler |
| Phân loại | Division Name | 3 | One-Hot |
| Văn bản | Review Text | 600 | TF-IDF (1–2 gram, stop-words tiếng Anh, chuẩn L2) |
| **Tổng** | | **606** | |

Số chiều TF-IDF được giới hạn ở 600 để `model_deep.json` đủ nhỏ cho việc nhúng
vào web tĩnh (tầng 1 có 600 × 128 trọng số).

In [6]:
NUMERIC = ["Age", "Rating", "Positive Feedback Count"]
DIVISION = "Division Name"
TEXT = "Review Text"

df[DIVISION] = df[DIVISION].fillna(df[DIVISION].mode()[0])

classes = list(dist.index)                     # thứ tự lớp theo tần suất giảm dần
class_index = {c: i for i, c in enumerate(classes)}
y = df[TARGET].map(class_index).values

idx_tmp, idx_test = train_test_split(np.arange(len(df)), test_size=0.15,
                                     random_state=SEED, stratify=y)
idx_train, idx_val = train_test_split(idx_tmp, test_size=0.1765,
                                      random_state=SEED, stratify=y[idx_tmp])
print(f"Train {len(idx_train)} | Val {len(idx_val)} | Test {len(idx_test)}")

# --- khối số: median CHỈ ước lượng trên train ---
num_raw = df[NUMERIC].astype(float)
num_median = num_raw.iloc[idx_train].median()
num_filled = num_raw.fillna(num_median)
num_scaler = StandardScaler().fit(num_filled.iloc[idx_train])

# --- khối one-hot Division ---
divisions = sorted(df[DIVISION].unique())
div_oh = np.eye(len(divisions))[[divisions.index(v) for v in df[DIVISION]]]
print("Division Name:", divisions)

# --- khối TF-IDF: fit CHỈ trên train ---
MAX_FEATURES = 600
tfidf = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1, 2),
                        stop_words="english", lowercase=True, sublinear_tf=False)
tfidf.fit(df[TEXT].iloc[idx_train])
print(f"Kích thước từ điển TF-IDF: {len(tfidf.vocabulary_)}")

def build(idx):
    n = num_scaler.transform(num_filled.iloc[idx])
    t = tfidf.transform(df[TEXT].iloc[idx]).toarray()
    return np.hstack([n, div_oh[idx], t])

Xtr, Xva, Xte = build(idx_train), build(idx_val), build(idx_test)
ytr, yva, yte = y[idx_train], y[idx_val], y[idx_test]
print(f"\nMa trận đặc trưng: {Xtr.shape} (3 số + {len(divisions)} one-hot + {MAX_FEATURES} TF-IDF)")

Train 16429 | Val 3522 | Test 3521
Division Name: ['General', 'General Petite', 'Initmates']


Kích thước từ điển TF-IDF: 600



Ma trận đặc trưng: (16429, 606) (3 số + 3 one-hot + 600 TF-IDF)


In [7]:
# 20 n-gram có trọng số IDF thấp nhất (phổ biến nhất) và cao nhất (đặc thù nhất)
vocab = np.array(tfidf.get_feature_names_out())
idf = tfidf.idf_
o = np.argsort(idf)
pd.DataFrame({
    "Phổ biến nhất (IDF thấp)": vocab[o[:15]],
    "IDF": idf[o[:15]].round(3),
    "Đặc thù nhất (IDF cao)": vocab[o[-15:]],
    "IDF ": idf[o[-15:]].round(3),
})

,Phổ biến nhất (IDF thấp),IDF,Đặc thù nhất (IDF cao),IDF
0,love,2.153,forward,5.817
1,size,2.269,perfect length,5.824
2,fit,2.329,home,5.824
3,dress,2.351,wear small,5.824
4,like,2.393,sad,5.839
5,wear,2.457,hold,5.839
6,great,2.499,smaller size,5.847
7,just,2.619,25,5.847
8,fabric,2.733,slight,5.863
9,color,2.751,stripes,5.863


## 3. Mạng nơ-ron sâu 5 tầng — head Softmax

```
Input(606) → [W1] 128 → ReLU → [W2] 64 → ReLU → [W3] 32 → ReLU
           → [W4] 16 → ReLU → [W5] 6 → Softmax   (loss = Categorical Cross-Entropy)
```

In [8]:
from __future__ import annotations

import numpy as np

# ----------------------------------------------------------------------------
# 1. Hàm kích hoạt và đạo hàm
# ----------------------------------------------------------------------------


def relu(z):
    """f(z) = max(0, z) — phá vỡ tính tuyến tính, giữ gradient không bão hoà ở nhánh dương."""
    return np.maximum(0.0, z)


def relu_grad(z):
    """f'(z) = 1 nếu z > 0, ngược lại 0."""
    return (z > 0).astype(z.dtype)


def sigmoid(z):
    """Ổn định số học: tách nhánh z >= 0 và z < 0 để tránh exp() tràn số."""
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out


def softmax(z):
    """Trừ max theo hàng trước khi exp — kỹ thuật log-sum-exp chống tràn số."""
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


# ----------------------------------------------------------------------------
# 2. Mạng nơ-ron sâu 5 tầng
# ----------------------------------------------------------------------------


class DeepMLP:
    """5-Layer MLP thuần NumPy với Adam, mini-batch, He-init, L2 và Dropout."""

    def __init__(
        self,
        input_dim: int,
        hidden=(128, 64, 32, 16),
        output_dim: int = 1,
        task: str = "binary",
        lr: float = 1e-3,
        l2: float = 1e-4,
        dropout: float = 0.0,
        seed: int = 42,
        class_weight=None,
    ):
        assert task in {"binary", "regression", "multiclass"}
        self.task = task
        # class_weight: vector trọng số theo lớp, dùng để chống mất cân bằng dữ liệu.
        # Mỗi mẫu được nhân thêm w[y] trong cả hàm mất mát lẫn gradient, tương đương
        # "nhân bản" mẫu của lớp hiếm mà không phải sao chép dữ liệu thật.
        self.class_weight = None if class_weight is None else np.asarray(class_weight, dtype=float)
        self.lr = lr
        self.l2 = l2
        self.dropout = dropout
        self.dims = [input_dim, *hidden, output_dim]
        self.rng = np.random.default_rng(seed)

        # --- Khởi tạo He (Kaiming): Var(W) = 2/fan_in, phù hợp với ReLU ---
        self.W, self.b = [], []
        for i in range(len(self.dims) - 1):
            fan_in, fan_out = self.dims[i], self.dims[i + 1]
            self.W.append(self.rng.normal(0.0, np.sqrt(2.0 / fan_in), (fan_in, fan_out)))
            self.b.append(np.zeros(fan_out))

        # --- Trạng thái Adam (moment bậc 1 và bậc 2) ---
        self.mW = [np.zeros_like(w) for w in self.W]
        self.vW = [np.zeros_like(w) for w in self.W]
        self.mb = [np.zeros_like(b) for b in self.b]
        self.vb = [np.zeros_like(b) for b in self.b]
        self.t = 0

        self.history = {"train_loss": [], "val_loss": [], "train_metric": [], "val_metric": []}

    # ----------------------------- forward ---------------------------------
    def forward(self, X, training: bool = False):
        """Trả về (output, cache). cache giữ A/Z của từng tầng để lan truyền ngược."""
        A = X
        As, Zs, masks = [A], [], []
        n_layers = len(self.W)

        for i in range(n_layers - 1):  # 4 tầng ẩn
            Z = A @ self.W[i] + self.b[i]
            A = relu(Z)
            if training and self.dropout > 0.0:
                # Inverted dropout: chia cho keep_prob ngay lúc train nên lúc
                # suy luận không cần chỉnh gì — trọng số xuất ra JSON dùng trực tiếp.
                keep = 1.0 - self.dropout
                mask = (self.rng.random(A.shape) < keep) / keep
                A = A * mask
                masks.append(mask)
            else:
                masks.append(None)
            Zs.append(Z)
            As.append(A)

        # Tầng 5 — output head
        Zout = A @ self.W[-1] + self.b[-1]
        Zs.append(Zout)
        if self.task == "binary":
            out = sigmoid(Zout)
        elif self.task == "multiclass":
            out = softmax(Zout)
        else:
            out = Zout
        As.append(out)
        return out, (As, Zs, masks)

    # ------------------------------ loss -----------------------------------
    def _sample_w(self, y_true):
        """Trọng số của từng mẫu suy ra từ class_weight (shape (n, 1))."""
        if self.class_weight is None or self.task != "multiclass":
            return None
        return self.class_weight[y_true.argmax(1)].reshape(-1, 1)

    def loss(self, y_pred, y_true):
        n = y_true.shape[0]
        if self.task == "binary":
            eps = 1e-12
            base = -np.mean(
                y_true * np.log(y_pred + eps) + (1 - y_true) * np.log(1 - y_pred + eps)
            )
        elif self.task == "multiclass":
            eps = 1e-12
            per_sample = -np.sum(y_true * np.log(y_pred + eps), axis=1, keepdims=True)
            w = self._sample_w(y_true)
            base = float(np.sum(per_sample if w is None else per_sample * w) / n)
        else:
            base = np.mean((y_pred - y_true) ** 2)
        reg = self.l2 * sum(np.sum(w * w) for w in self.W) / (2 * n)
        return base + reg

    # ---------------------------- backward ---------------------------------
    def backward(self, cache, y_true):
        As, Zs, masks = cache
        n = y_true.shape[0]
        n_layers = len(self.W)

        # Với cả 3 head, đạo hàm của loss theo pre-activation cuối cùng rút gọn
        # về (y_hat - y). Đây là lý do ta ghép Sigmoid/Softmax với Cross-Entropy
        # và Linear với MSE.
        dZ = (As[-1] - y_true) / n
        if self.task == "regression":
            dZ = 2.0 * dZ
        w = self._sample_w(y_true)
        if w is not None:
            dZ = dZ * w

        dW = [None] * n_layers
        db = [None] * n_layers

        for i in range(n_layers - 1, -1, -1):
            dW[i] = As[i].T @ dZ + self.l2 * self.W[i] / n
            db[i] = dZ.sum(axis=0)
            if i > 0:
                dA = dZ @ self.W[i].T
                if masks[i - 1] is not None:
                    dA = dA * masks[i - 1]
                dZ = dA * relu_grad(Zs[i - 1])
        return dW, db

    # ------------------------------ Adam -----------------------------------
    def _adam(self, dW, db, beta1=0.9, beta2=0.999, eps=1e-8):
        self.t += 1
        for i in range(len(self.W)):
            self.mW[i] = beta1 * self.mW[i] + (1 - beta1) * dW[i]
            self.vW[i] = beta2 * self.vW[i] + (1 - beta2) * (dW[i] ** 2)
            mhat = self.mW[i] / (1 - beta1**self.t)
            vhat = self.vW[i] / (1 - beta2**self.t)
            self.W[i] -= self.lr * mhat / (np.sqrt(vhat) + eps)

            self.mb[i] = beta1 * self.mb[i] + (1 - beta1) * db[i]
            self.vb[i] = beta2 * self.vb[i] + (1 - beta2) * (db[i] ** 2)
            mhat = self.mb[i] / (1 - beta1**self.t)
            vhat = self.vb[i] / (1 - beta2**self.t)
            self.b[i] -= self.lr * mhat / (np.sqrt(vhat) + eps)

    # ------------------------------ metric ---------------------------------
    def _metric(self, X, y):
        p, _ = self.forward(X, training=False)
        if self.task == "binary":
            return float(np.mean((p >= 0.5).astype(int) == y))
        if self.task == "multiclass":
            return float(np.mean(p.argmax(1) == y.argmax(1)))
        ss_res = np.sum((y - p) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return float(1 - ss_res / ss_tot)  # R^2

    # ------------------------------- fit -----------------------------------
    def fit(self, X, y, X_val=None, y_val=None, epochs=200, batch_size=32,
            verbose_every=20, patience=None):
        """Chu trình huấn luyện 4 bước: Forward -> Loss -> Backward -> Update."""
        n = X.shape[0]
        best_val, best_state, wait = np.inf, None, 0

        for ep in range(1, epochs + 1):
            idx = self.rng.permutation(n)
            Xs, ys = X[idx], y[idx]

            for s in range(0, n, batch_size):
                xb, yb = Xs[s:s + batch_size], ys[s:s + batch_size]
                _, cache = self.forward(xb, training=True)          # (1) Forward
                dW, db = self.backward(cache, yb)                   # (3) Backward
                self._adam(dW, db)                                  # (4) Update

            tr_pred, _ = self.forward(X, training=False)
            tr_loss = self.loss(tr_pred, y)                         # (2) Loss
            self.history["train_loss"].append(tr_loss)
            self.history["train_metric"].append(self._metric(X, y))

            if X_val is not None:
                va_pred, _ = self.forward(X_val, training=False)
                va_loss = self.loss(va_pred, y_val)
                self.history["val_loss"].append(va_loss)
                self.history["val_metric"].append(self._metric(X_val, y_val))

                if patience is not None:
                    if va_loss < best_val - 1e-6:
                        best_val, wait = va_loss, 0
                        best_state = ([w.copy() for w in self.W], [b.copy() for b in self.b])
                    else:
                        wait += 1
                        if wait >= patience:
                            if verbose_every:
                                print(f"  ⏹ Early stopping tại epoch {ep} (val_loss tốt nhất = {best_val:.4f})")
                            break

            if verbose_every and (ep % verbose_every == 0 or ep == 1):
                msg = f"  epoch {ep:4d} | train_loss={tr_loss:.4f} | train_metric={self.history['train_metric'][-1]:.4f}"
                if X_val is not None:
                    msg += f" | val_loss={self.history['val_loss'][-1]:.4f} | val_metric={self.history['val_metric'][-1]:.4f}"
                print(msg)

        if best_state is not None:
            self.W, self.b = best_state
        return self

    # ---------------------------- inference --------------------------------
    def predict_proba(self, X):
        out, _ = self.forward(X, training=False)
        return out

    def predict(self, X):
        out = self.predict_proba(X)
        if self.task == "binary":
            return (out >= 0.5).astype(int)
        if self.task == "multiclass":
            return out.argmax(1)
        return out

    # -------------------------- xuất ra JSON -------------------------------
    def n_params(self):
        return sum(w.size for w in self.W) + sum(b.size for b in self.b)

    def to_dict(self, decimals: int = 6):
        """Đóng gói trọng số về dict thuần Python để ghi ra model.json cho web."""
        return {
            "architecture": self.dims,
            "task": self.task,
            "activation": "relu",
            "n_params": int(self.n_params()),
            "layers": [
                {"W": np.round(w, decimals).tolist(), "b": np.round(bb, decimals).tolist()}
                for w, bb in zip(self.W, self.b)
            ],
        }


# ----------------------------------------------------------------------------
# 3. Tiện ích dùng chung
# ----------------------------------------------------------------------------


def one_hot(y, n_classes):
    out = np.zeros((len(y), n_classes))
    out[np.arange(len(y)), y] = 1.0
    return out


def forward_reference(model_dict, x):
    """Bản tham chiếu của thuật toán suy luận sẽ viết lại bằng JavaScript trên web.

    Dùng trong notebook để kiểm tra parity: NumPy <-> JSON <-> JavaScript.
    """
    a = np.asarray(x, dtype=float)
    layers = model_dict["layers"]
    for i, layer in enumerate(layers):
        z = a @ np.array(layer["W"]) + np.array(layer["b"])
        if i < len(layers) - 1:
            a = np.maximum(0.0, z)
        else:
            a = z
    if model_dict["task"] == "binary":
        return sigmoid(a)
    if model_dict["task"] == "multiclass":
        return softmax(a.reshape(1, -1))[0]
    return a

In [9]:
HP = dict(lr=1e-3, l2=1e-3, dropout=0.20, batch_size=128, epochs=120, patience=15)
print("Siêu tham số (chọn trên VALIDATION theo macro-F1):", HP)

n_classes = len(classes)
Ytr = one_hot(ytr, n_classes)
Yva = one_hot(yva, n_classes)

# Ba phương án xử lý mất cân bằng lớp:
#   - none      : không can thiệp
#   - sqrt      : làm mềm trọng số balanced bằng căn bậc hai (giảm cực đoan)
#   - balanced  : w_c = n / (C * n_c), đúng công thức 'balanced' của scikit-learn
counts = np.bincount(ytr, minlength=n_classes)
W_BAL = len(ytr) / (n_classes * counts)
W_SQRT = np.sqrt(W_BAL)
W_SQRT = W_SQRT / W_SQRT.mean() * 1.0
WEIGHT_SCHEMES = {"none": None, "sqrt": W_SQRT, "balanced": W_BAL}
print("\nTrọng số lớp theo từng phương án:")
print(pd.DataFrame({"balanced": W_BAL, "sqrt (đã chuẩn hoá)": W_SQRT},
                   index=classes).round(3).to_string())

reference_mlp = DeepMLP(input_dim=Xtr.shape[1], hidden=(128, 64, 32, 16), output_dim=n_classes,
                        task="multiclass", lr=HP["lr"], l2=HP["l2"], dropout=HP["dropout"],
                        seed=SEED)
mlp = reference_mlp

rows = []
names = ["Layer 1 (Input→H1)", "Layer 2 (H1→H2)", "Layer 3 (H2→H3)",
         "Layer 4 (H3→H4)", "Layer 5 (H4→Output)"]
acts = ["ReLU", "ReLU", "ReLU", "ReLU", "Softmax"]
for i, (nm, act) in enumerate(zip(names, acts)):
    fi, fo = mlp.dims[i], mlp.dims[i + 1]
    rows.append({"Tầng": nm, "W shape": f"({fi}, {fo})", "b shape": f"({fo},)",
                 "Output shape": f"(batch, {fo})", "Kích hoạt": act,
                 "Số tham số": fi * fo + fo})
shape_table = pd.DataFrame(rows)
shape_table.loc[len(shape_table)] = ["TỔNG", "", "", "", "", shape_table["Số tham số"].sum()]
shape_table

Siêu tham số (chọn trên VALIDATION theo macro-F1): {'lr': 0.001, 'l2': 0.001, 'dropout': 0.2, 'batch_size': 128, 'epochs': 120, 'patience': 15}

Trọng số lớp theo từng phương án:
          balanced  sqrt (đã chuẩn hoá)
Tops         0.374                0.316
Dresses      0.619                0.407
Bottoms      1.030                0.525
Intimate     2.254                0.776
Jackets      3.792                1.007
Trend       32.990                2.970


,Tầng,W shape,b shape,Output shape,Kích hoạt,Số tham số
0,Layer 1 (Input→H1),"(606, 128)","(128,)","(batch, 128)",ReLU,77696
1,Layer 2 (H1→H2),"(128, 64)","(64,)","(batch, 64)",ReLU,8256
2,Layer 3 (H2→H3),"(64, 32)","(32,)","(batch, 32)",ReLU,2080
3,Layer 4 (H3→H4),"(32, 16)","(16,)","(batch, 16)",ReLU,528
4,Layer 5 (H4→Output),"(16, 6)","(6,)","(batch, 6)",Softmax,102
5,TỔNG,,,,,88662


### 3.1. Ablation trọng số lớp — chọn phương án trên tập VALIDATION

Huấn luyện ba mạng **giống hệt nhau về kiến trúc và siêu tham số**, chỉ khác cách
đặt trọng số lớp, rồi chọn phương án tốt nhất theo **macro-F1 trên validation**
(macro-F1 coi mọi lớp quan trọng như nhau, nên đây mới là thước đo đúng cho dữ
liệu mất cân bằng — accuracy sẽ luôn thiên vị lớp "Tops").

In [10]:
variants, ablation = {}, []
t0 = time.perf_counter()
for scheme, w in WEIGHT_SCHEMES.items():
    m = DeepMLP(input_dim=Xtr.shape[1], hidden=(128, 64, 32, 16), output_dim=n_classes,
                task="multiclass", lr=HP["lr"], l2=HP["l2"], dropout=HP["dropout"],
                seed=SEED, class_weight=w)
    m.fit(Xtr, Ytr, Xva, Yva, epochs=HP["epochs"], batch_size=HP["batch_size"],
          verbose_every=0, patience=HP["patience"])
    pv = m.predict_proba(Xva).argmax(1)
    variants[scheme] = m
    ablation.append({
        "Trọng số lớp": scheme,
        "Epoch": len(m.history["train_loss"]),
        "Val Accuracy": accuracy_score(yva, pv),
        "Val Balanced Acc": balanced_accuracy_score(yva, pv),
        "Val Macro-F1": f1_score(yva, pv, average="macro", zero_division=0),
    })
numpy_time = time.perf_counter() - t0

ablation_df = pd.DataFrame(ablation).set_index("Trọng số lớp").round(4)
BEST_SCHEME = ablation_df["Val Macro-F1"].idxmax()
mlp = variants[BEST_SCHEME]
CLASS_W = WEIGHT_SCHEMES[BEST_SCHEME]
print(f"⏱ Tổng thời gian huấn luyện 3 biến thể (NumPy): {numpy_time:.2f}s | "
      f"Tham số mỗi mạng: {mlp.n_params():,}")
print(f"=> Phương án được chọn theo Val Macro-F1: '{BEST_SCHEME}'\n")
ablation_df

⏱ Tổng thời gian huấn luyện 3 biến thể (NumPy): 39.36s | Tham số mỗi mạng: 88,662
=> Phương án được chọn theo Val Macro-F1: 'sqrt'



,Epoch,Val Accuracy,Val Balanced Acc,Val Macro-F1
Trọng số lớp,,,,
none,19,0.8339,0.6147,0.6413
sqrt,19,0.8206,0.6586,0.6583
balanced,19,0.7541,0.6413,0.6259


## 4. Bản PyTorch tương đương

In [11]:
import torch
import torch.nn as nn

torch.manual_seed(SEED)
D = HP["dropout"]
torch_net = nn.Sequential(
    nn.Linear(Xtr.shape[1], 128), nn.ReLU(), nn.Dropout(D),
    nn.Linear(128, 64), nn.ReLU(), nn.Dropout(D),
    nn.Linear(64, 32), nn.ReLU(), nn.Dropout(D),
    nn.Linear(32, 16), nn.ReLU(), nn.Dropout(D),
    nn.Linear(16, n_classes),
)
WEIGHT_DECAY = HP["l2"] / HP["batch_size"]
opt = torch.optim.Adam(torch_net.parameters(), lr=HP["lr"], weight_decay=WEIGHT_DECAY)
lossfn = nn.CrossEntropyLoss(
    weight=None if CLASS_W is None else torch.tensor(CLASS_W, dtype=torch.float32))

Xtr_t = torch.tensor(Xtr, dtype=torch.float32); ytr_t = torch.tensor(ytr, dtype=torch.long)
Xva_t = torch.tensor(Xva, dtype=torch.float32); yva_t = torch.tensor(yva, dtype=torch.long)
Xte_t = torch.tensor(Xte, dtype=torch.float32)

torch_hist = {"train_loss": [], "val_loss": []}
best_state, best_val, wait = None, np.inf, 0
t0 = time.perf_counter()
n = len(Xtr_t)
for ep in range(1, HP["epochs"] + 1):
    torch_net.train()
    perm = torch.randperm(n)
    for s in range(0, n, HP["batch_size"]):
        i = perm[s:s + HP["batch_size"]]
        opt.zero_grad()
        lossfn(torch_net(Xtr_t[i]), ytr_t[i]).backward()
        opt.step()
    torch_net.eval()
    with torch.no_grad():
        tl = lossfn(torch_net(Xtr_t), ytr_t).item()
        vl = lossfn(torch_net(Xva_t), yva_t).item()
    torch_hist["train_loss"].append(tl); torch_hist["val_loss"].append(vl)
    if vl < best_val - 1e-7:
        best_val, wait = vl, 0
        best_state = {k: v.clone() for k, v in torch_net.state_dict().items()}
    else:
        wait += 1
        if wait >= HP["patience"]:
            print(f"  ⏹ Early stopping tại epoch {ep}")
            break
    if ep % 10 == 0:
        print(f"  epoch {ep:3d} | train_ce={tl:.4f} | val_ce={vl:.4f}")
torch_net.load_state_dict(best_state)
torch_time = time.perf_counter() - t0
print(f"\n⏱ PyTorch: {torch_time:.2f}s")

  epoch  10 | train_ce=0.4572 | val_ce=0.7619


  epoch  20 | train_ce=0.2155 | val_ce=1.1370


  ⏹ Early stopping tại epoch 21

⏱ PyTorch: 14.12s


## 5. Baseline Học máy truyền thống

In [12]:
classical = {
    "Logistic Regression": LogisticRegression(max_iter=1500, random_state=SEED),
    "Multinomial NB": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(max_depth=14, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=120, max_depth=22,
                                            n_jobs=-1, random_state=SEED),
}
# MultinomialNB đòi hỏi đầu vào không âm -> dùng bản chưa chuẩn hoá cho khối số
Xtr_nb = np.hstack([num_filled.iloc[idx_train].values, div_oh[idx_train],
                    tfidf.transform(df[TEXT].iloc[idx_train]).toarray()])
Xva_nb = np.hstack([num_filled.iloc[idx_val].values, div_oh[idx_val],
                    tfidf.transform(df[TEXT].iloc[idx_val]).toarray()])
Xte_nb = np.hstack([num_filled.iloc[idx_test].values, div_oh[idx_test],
                    tfidf.transform(df[TEXT].iloc[idx_test]).toarray()])

fitted = {}
for name, m in classical.items():
    t = time.perf_counter()
    if name == "Multinomial NB":
        m.fit(Xtr_nb, ytr)
        acc = accuracy_score(yva, m.predict(Xva_nb))
    else:
        m.fit(Xtr, ytr)
        acc = accuracy_score(yva, m.predict(Xva))
    fitted[name] = m
    print(f"{name:22s} val acc = {acc:.4f}   ({time.perf_counter()-t:.1f}s)")

Logistic Regression    val acc = 0.8450   (71.9s)


Multinomial NB         val acc = 0.7195   (0.7s)


Decision Tree          val acc = 0.8032   (2.3s)


Random Forest          val acc = 0.8186   (1.5s)


## 6. Đánh giá trên tập kiểm thử

In [13]:
proba = {}
for name, m in fitted.items():
    proba[name] = m.predict_proba(Xte_nb if name == "Multinomial NB" else Xte)
for scheme, m in variants.items():
    if scheme != BEST_SCHEME:
        proba[f"Deep MLP-5 (w={scheme})"] = m.predict_proba(Xte)
proba["Deep MLP-5 (NumPy)"] = mlp.predict_proba(Xte)
with torch.no_grad():
    proba["Deep MLP-5 (PyTorch)"] = torch.softmax(torch_net(Xte_t), dim=1).numpy()

def evaluate(name, P):
    pred = P.argmax(1)
    return {
        "Mô hình": name,
        "Accuracy": accuracy_score(yte, pred),
        "Balanced Acc": balanced_accuracy_score(yte, pred),
        "Macro-F1": f1_score(yte, pred, average="macro", zero_division=0),
        "Weighted-F1": f1_score(yte, pred, average="weighted", zero_division=0),
        "Top-2 Acc": top_k_accuracy_score(yte, P, k=2, labels=np.arange(n_classes)),
    }

res_df = pd.DataFrame([evaluate(k, v) for k, v in proba.items()]).set_index("Mô hình").round(4)
baseline = pd.Series(yte).value_counts(normalize=True).max()
print(f"Đường cơ sở 'đoán lớp phổ biến nhất' (Tops): accuracy = {baseline:.4f}\n")
res_df.sort_values("Macro-F1", ascending=False)

Đường cơ sở 'đoán lớp phổ biến nhất' (Tops): accuracy = 0.4459



,Accuracy,Balanced Acc,Macro-F1,Weighted-F1,Top-2 Acc
Mô hình,,,,,
Logistic Regression,0.8481,0.6334,0.6615,0.8428,0.9341
Deep MLP-5 (PyTorch),0.8270,0.6552,0.6588,0.8270,0.9179
Deep MLP-5 (NumPy),0.8242,0.6614,0.6576,0.8254,0.9270
Decision Tree,0.8185,0.6178,0.6474,0.8140,0.8915
Deep MLP-5 (w=none),0.8378,0.6171,0.6407,0.8309,0.9341
Deep MLP-5 (w=balanced),0.7478,0.6601,0.6268,0.7749,0.8344
Random Forest,0.8273,0.5686,0.5993,0.8116,0.9284
Multinomial NB,0.7060,0.4772,0.4956,0.6836,0.8895


In [14]:
pred_np = proba["Deep MLP-5 (NumPy)"].argmax(1)
print("Báo cáo chi tiết — Deep MLP-5 (NumPy from scratch)\n")
print(classification_report(yte, pred_np, target_names=[VI_NAME[c] for c in classes],
                            digits=4, zero_division=0))

Báo cáo chi tiết — Deep MLP-5 (NumPy from scratch)

                         precision    recall  f1-score   support

              Áo (Tops)     0.8513    0.8350    0.8431      1570
      Váy đầm (Dresses)     0.9172    0.8534    0.8842       948
         Quần (Bottoms)     0.6681    0.8158    0.7346       570
Đồ mặc trong (Intimate)     0.9738    0.8577    0.9121       260
     Áo khoác (Jackets)     0.5402    0.6065    0.5714       155
  Hàng xu hướng (Trend)     0.0000    0.0000    0.0000        18

               accuracy                         0.8242      3521
              macro avg     0.6584    0.6614    0.6576      3521
           weighted avg     0.8304    0.8242    0.8254      3521



## 7. Trực quan hoá

### 7.1. Đường cong huấn luyện Cross-Entropy

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

ep = range(1, len(mlp.history["train_loss"]) + 1)
axes[0].plot(ep, mlp.history["train_loss"], color="#2563eb", lw=2, label="Train CE")
axes[0].plot(ep, mlp.history["val_loss"], color="#f97316", lw=2, label="Validation CE")
best_ep = int(np.argmin(mlp.history["val_loss"])) + 1
axes[0].axvline(best_ep, ls="--", color="#16a34a")
axes[0].axhline(np.log(n_classes), color="#9ca3af", ls=":",
                label=f"Đoán ngẫu nhiên = ln(6) = {np.log(n_classes):.3f}")
axes[0].annotate(f"epoch {best_ep}", xy=(best_ep, min(mlp.history["val_loss"])),
                 xytext=(best_ep + 6, min(mlp.history["val_loss"]) + 0.15),
                 arrowprops=dict(arrowstyle="->", color="#16a34a"), fontsize=8, color="#16a34a")
axes[0].set_title("(a) Categorical Cross-Entropy — NumPy", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(fontsize=8)

axes[1].plot(ep, mlp.history["train_metric"], color="#2563eb", lw=2, label="Train accuracy")
axes[1].plot(ep, mlp.history["val_metric"], color="#f97316", lw=2, label="Validation accuracy")
axes[1].axhline(baseline, color="#9ca3af", ls=":", label=f"Đoán lớp đa số = {baseline:.3f}")
axes[1].set_title("(b) Accuracy theo epoch", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy"); axes[1].legend(fontsize=8)

ep2 = range(1, len(torch_hist["train_loss"]) + 1)
axes[2].plot(ep2, torch_hist["train_loss"], color="#7c3aed", lw=2, label="Train CE (PyTorch)")
axes[2].plot(ep2, torch_hist["val_loss"], color="#dc2626", lw=2, label="Val CE (PyTorch)")
axes[2].set_title("(c) Bản PyTorch — cùng kiến trúc", fontweight="bold")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Loss"); axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIG / "p3_fig2_curves.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_20440\3633226308.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 7.2. Ma trận nhầm lẫn đa lớp

In [16]:
short = [c for c in classes]
fig, axes = plt.subplots(1, 2, figsize=(15, 5.6))

cm = confusion_matrix(yte, pred_np, labels=np.arange(n_classes))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0], cbar=False,
            xticklabels=short, yticklabels=short, annot_kws={"size": 9})
axes[0].set_title("(a) Số lượng tuyệt đối — Deep MLP-5", fontweight="bold")
axes[0].set_xlabel("Dự đoán"); axes[0].set_ylabel("Thực tế"); axes[0].grid(False)

cmn = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cmn, annot=True, fmt=".2f", cmap="RdYlGn", ax=axes[1], cbar=True,
            vmin=0, vmax=1, xticklabels=short, yticklabels=short, annot_kws={"size": 9})
axes[1].set_title("(b) Chuẩn hoá theo hàng (recall từng lớp)", fontweight="bold")
axes[1].set_xlabel("Dự đoán"); axes[1].set_ylabel("Thực tế"); axes[1].grid(False)

plt.tight_layout()
plt.savefig(FIG / "p3_fig3_confusion.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_20440\2368115933.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 7.3. F1 theo từng lớp và biểu đồ đối sánh

In [17]:
fig, axes = plt.subplots(1, 2, figsize=(15.5, 5))

order = list(proba.keys())
per_class = pd.DataFrame(
    {name: f1_score(yte, proba[name].argmax(1), average=None,
                    labels=np.arange(n_classes), zero_division=0) for name in order},
    index=short,
)
test_support = pd.Series(yte).value_counts().reindex(range(n_classes)).values
per_class.plot(kind="bar", ax=axes[0], width=0.78, colormap="tab10")
axes[0].set_title("(a) F1 theo từng lớp — lớp hiếm là điểm yếu chung", fontweight="bold")
axes[0].set_ylabel("F1"); axes[0].set_xlabel(""); axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(fontsize=7, ncol=2, loc="lower left", framealpha=.92)
axes[0].set_xticklabels([f"{c}\n(n={n} test)" for c, n in zip(short, test_support)], fontsize=8)
axes[0].set_ylim(0, 1.0)

metrics = ["Accuracy", "Balanced Acc", "Macro-F1", "Top-2 Acc"]
x = np.arange(len(order)); w = 0.2
for i, m in enumerate(metrics):
    vals = [res_df.loc[o, m] for o in order]
    bars = axes[1].bar(x + (i - 1.5) * w, vals, w, label=m)
    for b, v in zip(bars, vals):
        axes[1].text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.2f}",
                     ha="center", fontsize=6.5, rotation=90)
axes[1].axhline(baseline, color="#9ca3af", ls=":", label="Đoán lớp đa số")
axes[1].set_xticks(x)
axes[1].set_xticklabels([o.replace(" (", "\n(") for o in order], fontsize=7.5,
                        rotation=18, ha="right")
axes[1].set_ylim(0, 1.18); axes[1].set_title("(b) Đối sánh toàn cục", fontweight="bold")
axes[1].legend(fontsize=7.5, ncol=2, loc="upper left", framealpha=.92)

plt.tight_layout()
plt.savefig(FIG / "p3_fig4_perclass.png", bbox_inches="tight")
plt.show()
per_class.round(3)

C:\Users\admin\AppData\Local\Temp\ipykernel_20440\4006607349.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Logistic Regression,Multinomial NB,Decision Tree,Random Forest,Deep MLP-5 (w=none),Deep MLP-5 (w=balanced),Deep MLP-5 (NumPy),Deep MLP-5 (PyTorch)
Tops,0.861,0.755,0.834,0.845,0.855,0.778,0.843,0.845
Dresses,0.896,0.667,0.873,0.889,0.885,0.836,0.884,0.877
Bottoms,0.787,0.611,0.711,0.730,0.774,0.701,0.735,0.750
Intimate,0.923,0.917,0.918,0.923,0.918,0.891,0.912,0.922
Jackets,0.502,0.025,0.549,0.209,0.412,0.532,0.571,0.559
Trend,0.000,0.000,0.000,0.000,0.000,0.023,0.000,0.000


### 7.4. Mạng đã học được gì? — Các n-gram kích hoạt mạnh nhất mỗi lớp

Với mạng sâu ta không có "hệ số" như hồi quy tuyến tính. Thay vào đó dùng
**gradient của logit lớp c theo đầu vào** (saliency) — đo mức độ một n-gram
đẩy dự đoán về phía lớp đó.

Một lưu ý quan trọng: gradient thô $\partial\,\text{logit}_c/\partial x$ chứa cả
thành phần "chung cho mọi lớp" (những từ chỉ báo rằng câu này *là* một bình luận
về quần áo). Để lấy phần **đặc trưng riêng của lớp**, ta trừ đi trung bình
saliency trên toàn bộ 6 lớp — đúng như Softmax làm khi chuẩn hoá các logit.

In [18]:
def class_saliency(model, class_id, x_base):
    """d(logit_c)/d(x) tại điểm x_base, tính bằng chain rule thủ công."""
    A = x_base.reshape(1, -1)
    Zs = []
    for i in range(len(model.W) - 1):
        Z = A @ model.W[i] + model.b[i]
        A = relu(Z)
        Zs.append(Z)
    g = model.W[-1][:, class_id]                       # d logit / d A4
    for i in range(len(model.W) - 2, -1, -1):
        g = g * relu_grad(Zs[i]).ravel()               # qua ReLU
        g = model.W[i] @ g                             # qua W
    return g

x_base = Xtr.mean(axis=0)
text_offset = len(NUMERIC) + len(divisions)
G = np.vstack([class_saliency(mlp, ci, x_base)[text_offset:] for ci in range(n_classes)])
G_rel = G - G.mean(axis=0, keepdims=True)              # phần riêng của từng lớp

top_terms = {}
for ci, c in enumerate(classes):
    top = np.argsort(-G_rel[ci])[:8]
    top_terms[VI_NAME[c]] = [f"{vocab[j]} ({G_rel[ci, j]:+.2f})" for j in top]
pd.DataFrame(top_terms)

,Áo (Tops),Váy đầm (Dresses),Quần (Bottoms),Đồ mặc trong (Intimate),Áo khoác (Jackets),Hàng xu hướng (Trend)
0,shirt (+16.23),dress (+30.12),skirt (+5.58),dress (+10.90),shirt (+9.69),dress (+10.58)
1,tops (+14.48),dresses (+13.76),pants (+4.33),love dress (+4.80),tops (+9.50),dresses (+4.64)
2,blouse (+13.69),love dress (+13.13),pilcro (+4.11),jacket (+4.69),blouse (+8.57),love dress (+4.58)
3,sweater (+12.62),wedding (+10.86),jumpsuit (+3.97),dresses (+4.64),jacket (+7.77),wedding (+3.08)
4,tank (+11.02),knee (+7.72),shorts (+3.93),slip (+2.90),sweater (+7.45),slip (+2.83)
5,cami (+8.67),slip (+7.71),stretch (+3.91),coat (+2.87),coat (+7.37),knee (+2.52)
6,love shirt (+8.60),belt (+7.54),leg (+3.82),wedding (+2.47),tank (+6.76),beautiful dress (+2.41)
7,boxy (+7.66),knees (+7.39),thighs (+3.44),light (+2.24),vest (+6.26),boots (+2.26)


## 8. Xuất mô hình cho web app

In [19]:
deep_bundle = {
    **mlp.to_dict(decimals=5),
    "app": "customer_interest",
    "model_name": "Deep MLP 5 tầng (NumPy from scratch)",
    "classes": classes,
    "class_labels_vi": [VI_NAME[c] for c in classes],
    "numeric": {
        "names": NUMERIC,
        "fill": [float(num_median[c]) for c in NUMERIC],
        "mean": num_scaler.mean_.tolist(),
        "scale": num_scaler.scale_.tolist(),
    },
    "division": {"name": DIVISION, "categories": divisions,
                 "fill": str(df[DIVISION].mode()[0])},
    "text": {
        "column": TEXT,
        "vocab": vocab.tolist(),
        "idf": np.round(idf, 6).tolist(),
        "ngram_range": [1, 2],
        "lowercase": True,
        "stop_words": sorted(ENGLISH_STOP_WORDS),
    },
    "metrics": {k.lower().replace(" ", "_").replace("-", "_"):
                float(res_df.loc["Deep MLP-5 (NumPy)", k]) for k in res_df.columns},
    "baseline_metrics": {name: {k: float(res_df.loc[name, k]) for k in res_df.columns}
                         for name in classical},
    "majority_baseline": float(baseline),
    "training": {
        "epochs_run": len(mlp.history["train_loss"]),
        **{k: HP[k] for k in ["lr", "l2", "dropout", "batch_size"]},
        "class_weight_scheme": BEST_SCHEME,
        "class_weight": None if CLASS_W is None else [round(float(w), 4) for w in CLASS_W],
        "numpy_seconds": round(numpy_time, 2),
        "pytorch_seconds": round(torch_time, 2),
        "n_train": int(len(idx_train)), "n_val": int(len(idx_val)), "n_test": int(len(idx_test)),
    },
    "history": {
        "train_loss": [round(v, 5) for v in mlp.history["train_loss"]],
        "val_loss": [round(v, 5) for v in mlp.history["val_loss"]],
        "train_acc": [round(v, 5) for v in mlp.history["train_metric"]],
        "val_acc": [round(v, 5) for v in mlp.history["val_metric"]],
    },
}

out_path = ROOT / "public" / "model_deep.json"
out_path.parent.mkdir(exist_ok=True)
out_path.write_text(json.dumps(deep_bundle, ensure_ascii=False, separators=(",", ":")),
                    encoding="utf-8")
print(f"✅ Đã ghi {out_path} — {out_path.stat().st_size/1024:.1f} KB")

✅ Đã ghi C:\Users\admin\Downloads\bt-thay quế\tuan 2\customer-behavior-predict\public\model_deep.json — 741.9 KB


### 8.1. Kiểm tra parity NumPy ↔ JSON

In [20]:
reloaded = json.loads(out_path.read_text(encoding="utf-8"))
P_ref = proba["Deep MLP-5 (NumPy)"]
max_err, mism = 0.0, 0
for i in range(0, len(Xte), 7):          # lấy mẫu ~1/7 tập test cho nhanh
    p = forward_reference(reloaded, Xte[i])
    max_err = max(max_err, float(np.max(np.abs(p - P_ref[i]))))
    mism += int(np.argmax(p) != np.argmax(P_ref[i]))
print(f"Sai lệch xác suất lớn nhất : {max_err:.3e}")
print(f"Số mẫu lệch nhãn dự đoán   : {mism}")
assert max_err < 1e-3 and mism == 0
print("✅ Parity PASSED — làm tròn 5 chữ số thập phân không đổi kết quả dự đoán.")

Sai lệch xác suất lớn nhất : 6.789e-05
Số mẫu lệch nhãn dự đoán   : 0
✅ Parity PASSED — làm tròn 5 chữ số thập phân không đổi kết quả dự đoán.


### 8.2. Bản tham chiếu suy luận cho JavaScript

Ghi thêm một vài mẫu kiểm thử để `ml/parity.mjs` đối chiếu code JS với NumPy.

In [21]:
samples = []
for i in idx_test[:25]:
    samples.append({
        "input": {
            "Age": int(df["Age"].iloc[i]),
            "Rating": int(df["Rating"].iloc[i]),
            "Positive Feedback Count": int(df["Positive Feedback Count"].iloc[i]),
            "Division Name": str(df[DIVISION].iloc[i]),
            "Review Text": str(df[TEXT].iloc[i]),
        },
        "expected_proba": [round(float(v), 6) for v in mlp.predict_proba(build([i]))[0]],
    })
(ROOT / "ml" / "deep_parity_samples.json").write_text(
    json.dumps(samples, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"✅ Đã ghi {len(samples)} mẫu tham chiếu vào ml/deep_parity_samples.json")

✅ Đã ghi 25 mẫu tham chiếu vào ml/deep_parity_samples.json


## 9. Kết luận Bài toán 3

In [22]:
res_df.sort_values("Macro-F1", ascending=False)

,Accuracy,Balanced Acc,Macro-F1,Weighted-F1,Top-2 Acc
Mô hình,,,,,
Logistic Regression,0.8481,0.6334,0.6615,0.8428,0.9341
Deep MLP-5 (PyTorch),0.8270,0.6552,0.6588,0.8270,0.9179
Deep MLP-5 (NumPy),0.8242,0.6614,0.6576,0.8254,0.9270
Decision Tree,0.8185,0.6178,0.6474,0.8140,0.8915
Deep MLP-5 (w=none),0.8378,0.6171,0.6407,0.8309,0.9341
Deep MLP-5 (w=balanced),0.7478,0.6601,0.6268,0.7749,0.8344
Random Forest,0.8273,0.5686,0.5993,0.8116,0.9284
Multinomial NB,0.7060,0.4772,0.4956,0.6836,0.8895


In [23]:
print(f"""
TÓM TẮT BÀI TOÁN 3 — TIKI INTEREST PREDICTION (6 lớp)
{'='*68}
Kiến trúc      : {Xtr.shape[1]} → 128 → 64 → 32 → 16 → {n_classes} (Softmax)
Tổng tham số   : {mlp.n_params():,}
Epoch đã chạy  : {len(mlp.history['train_loss'])} (early stopping, patience={HP['patience']})
Trọng số lớp   : {BEST_SCHEME} (chọn theo macro-F1 trên validation)
Thời gian train: NumPy {numpy_time:.2f}s cho 3 biến thể | PyTorch {torch_time:.2f}s
Đường cơ sở    : {baseline:.4f} (đoán lớp đa số 'Tops')
Test Accuracy  : {res_df.loc['Deep MLP-5 (NumPy)', 'Accuracy']:.4f}
Balanced Acc   : {res_df.loc['Deep MLP-5 (NumPy)', 'Balanced Acc']:.4f}
Macro-F1       : {res_df.loc['Deep MLP-5 (NumPy)', 'Macro-F1']:.4f}
Top-2 Accuracy : {res_df.loc['Deep MLP-5 (NumPy)', 'Top-2 Acc']:.4f}
{'='*68}
""")


TÓM TẮT BÀI TOÁN 3 — TIKI INTEREST PREDICTION (6 lớp)
Kiến trúc      : 606 → 128 → 64 → 32 → 16 → 6 (Softmax)
Tổng tham số   : 88,662
Epoch đã chạy  : 19 (early stopping, patience=15)
Trọng số lớp   : sqrt (chọn theo macro-F1 trên validation)
Thời gian train: NumPy 39.36s cho 3 biến thể | PyTorch 14.12s
Đường cơ sở    : 0.4459 (đoán lớp đa số 'Tops')
Test Accuracy  : 0.8242
Balanced Acc   : 0.6614
Macro-F1       : 0.6576
Top-2 Accuracy : 0.9270

